# Phase 1 — INT8 Baseline Reproduction

**Project**: Quantum-Inspired Tensor-Network Compression of OpenVLA-7B  
**Challenge**: Global Quantum + AI Challenge 2026 — VW Group Enterprise Track  
**Phase**: 1 of 7 | **Execution**: Google Colab (GPU runtime required)

---

## What this notebook does

Loads OpenVLA-7B in INT8 via bitsandbytes (the accepted baseline per challenge §5.4),
runs inference on 200 held-out Open X-Embodiment `bridge_dataset` episodes for each
of the three evaluation seeds, and writes `results/baseline_metrics.json` with all
§5.5 benchmark fields.

**Metrics recorded** (mean ± std, 3 independent runs):
- Action-prediction L1 error
- Per-sample inference time (ms)
- Peak GPU memory (MiB)
- GPU power draw (W) and total energy (kWh)
- Total parameter count

## Before running

1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU or A100).
2. Add two secrets in Colab Secrets (left panel → key icon):
   - `HF_TOKEN` — HuggingFace token with read access to `openvla/openvla-7b`
     (accept the [LLaMA-2 Community License](https://huggingface.co/meta-llama/Llama-2-7b)
     and the [OpenVLA model agreement](https://huggingface.co/openvla/openvla-7b) first)
   - `GH_TOKEN` — GitHub Personal Access Token with `repo` scope
     (Settings → Developer settings → Personal access tokens → Tokens (classic))
3. **Run cells in order from the top.** Do not skip or reorder.

---

> **License notice** — OpenVLA-7B *code* is MIT.
> The *model weights* are subject to the **LLaMA-2 Community License**
> (non-commercial research use only). Downloading the weights via HuggingFace Hub
> constitutes acceptance of that license.

In [ ]:
# ── Cell 1: Install / pin dependencies ───────────────────────────────────────
# Uses subprocess so pip runs in the current Python env before any imports.
#
# Version pins are taken directly from openvla/openvla pyproject.toml /
# requirements-min.txt (github.com/openvla/openvla).  Exact pins matter:
#   transformers==4.40.1  — the only version tested by the OpenVLA authors;
#                           4.40.2+ and 5.x break modeling_prismatic.py
#   tokenizers==0.19.1   — must match transformers 4.40.1 (HF ecosystem coupling)
#   timm==0.9.10         — imported by modeling_prismatic.py via trust_remote_code
#   sentencepiece==0.1.99 — required by the LLaMA-2 tokenizer in the backbone
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# Install transformers and its tightly-coupled companions first so that
# pip resolves their mutual constraints before the rest of the stack.
_pip(
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "sentencepiece==0.1.99",
)
_pip(
    "accelerate>=0.27.0",
    "bitsandbytes>=0.43.0",
    "pynvml",
    "tensorflow-datasets",
    "PyYAML",
    "tqdm",
)
print("pip installs complete.")

In [ ]:
# ── Cell 2: Verify transformers version and key imports ──────────────────────
# Fails loudly before the 15 GB model download if the wrong version landed.
# If this cell raises, do: Runtime → Disconnect and delete runtime, then
# re-run from Cell 1 without running anything else first.
import transformers

REQUIRED_TRANSFORMERS = "4.40.1"
_ver = transformers.__version__
print(f"transformers : {_ver}")

if _ver != REQUIRED_TRANSFORMERS:
    raise RuntimeError(
        f"transformers {_ver} is installed but OpenVLA requires exactly {REQUIRED_TRANSFORMERS}.\n"
        f"Other 4.x builds and all 5.x builds break modeling_prismatic.py\n"
        f"(_supports_sdpa missing, is_flax_available moved, etc.).\n\n"
        f"Fix:\n"
        f"  1. Runtime → Disconnect and delete runtime\n"
        f"  2. Re-open the notebook and run Cell 1 before anything else\n"
        f"  3. Do NOT run any other cells before Cell 1 finishes"
    )
print(f"  ✓  {REQUIRED_TRANSFORMERS} confirmed")

try:
    from transformers import AutoModelForVision2Seq, AutoProcessor
    print("AutoModelForVision2Seq : importable ✓")
    print("AutoProcessor          : importable ✓")
except ImportError as exc:
    raise ImportError(
        f"transformers {_ver} installed but import failed: {exc}\n"
        "Check pip output in Cell 1 for conflicts."
    )

In [ ]:
# ── Cell 3: Clone repo and install the vlam_compress package ─────────────────
# The repo is private — authenticates via the GH_TOKEN Colab Secret.
# Add a GitHub PAT (repo scope) as GH_TOKEN in Colab Secrets before running.
import os, sys, subprocess

REPO_DIR = "/content/vlam"

try:
    from google.colab import userdata
    _gh_token = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_gh_token}@github.com/quantumadventurer11/vw-quantum-vlam-challenge"
    print("GH_TOKEN loaded from Colab Secrets.")
except Exception as _e:
    print(f"GH_TOKEN not found in secrets ({_e}).")
    print("Falling back to unauthenticated URL — will prompt for credentials.")
    REPO_URL = "https://github.com/quantumadventurer11/vw-quantum-vlam-challenge"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print(f"Working directory : {os.getcwd()}")
print("vlam_compress package installed.")

In [ ]:
# ── Cell 3: HuggingFace authentication ───────────────────────────────────────
# Requires: HF_TOKEN secret set in Colab Secrets (left panel → key icon)
# Token must have read access to openvla/openvla-7b (gated repo).
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    login(token=token, add_to_git_credential=False)
    print("Logged in via Colab Secrets.")
except Exception as e:
    print(f"Secret not found ({e}). Falling back to interactive login...")
    login()

In [ ]:
# ── Cell 4: Google Cloud authentication (needed for GCS dataset access) ───────
from google.colab import auth
auth.authenticate_user()
print("Google authentication complete.")

In [ ]:
# ── Cell 5: Imports ───────────────────────────────────────────────────────────
import json
import platform
import time
from pathlib import Path

import numpy as np
import pynvml
import torch
import transformers
import yaml
from PIL import Image
from tqdm.auto import tqdm

import bitsandbytes as bnb  # noqa: F401  (confirms install)
import tensorflow as tf
import tensorflow_datasets as tfds

print("All imports OK.")

In [ ]:
# ── Cell 6: Environment diagnostics (logged to results for §6 Resource Declaration)
import bitsandbytes

assert torch.cuda.is_available(), "No GPU found — change runtime to GPU and restart."

props = torch.cuda.get_device_properties(0)
print(f"PyTorch          : {torch.__version__}")
print(f"CUDA             : {torch.version.cuda}")
print(f"Transformers     : {transformers.__version__}")
print(f"bitsandbytes     : {bitsandbytes.__version__}")
print(f"TensorFlow       : {tf.__version__}")
print(f"GPU name         : {props.name}")
print(f"GPU VRAM total   : {props.total_memory / 1024**3:.1f} GB")
print(f"GPU multi-proc   : {props.multi_processor_count}")
print(f"Platform         : {platform.platform()}")

In [ ]:
# ── Cell 7: Load project config ───────────────────────────────────────────────
with open("configs/seeds.yaml") as f:
    cfg = yaml.safe_load(f)

SEEDS            = cfg["seeds"]               # [42, 1337, 2024]
N_EVAL_EPISODES  = cfg["eval"]["n_episodes"]  # 200
MODEL_ID         = "openvla/openvla-7b"
UNNORM_KEY       = "bridge_orig"              # BridgeV2 action normalization stats
GCS_DATASET_PATH = "gs://gresearch/robotics/bridge_dataset/1.0.0/"
RESULTS_DIR      = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Seeds            : {SEEDS}")
print(f"Episodes per run : {N_EVAL_EPISODES}")
print(f"Model            : {MODEL_ID}")
print(f"Dataset path     : {GCS_DATASET_PATH}")

In [ ]:
# ── Cell 8: Initialize pynvml (GPU power monitoring) ─────────────────────────
pynvml.nvmlInit()
gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)

driver_ver = pynvml.nvmlSystemGetDriverVersion()
if isinstance(driver_ver, bytes):
    driver_ver = driver_ver.decode()

print(f"GPU name (nvml)  : {pynvml.nvmlDeviceGetName(gpu_handle)}")
print(f"Driver version   : {driver_ver}")
print(f"Current power    : {pynvml.nvmlDeviceGetPowerUsage(gpu_handle) / 1000:.1f} W")

In [ ]:
# ── Cell 9: Load OpenVLA-7B in INT8 ──────────────────────────────────────────
# Weights are subject to the LLaMA-2 Community License (non-commercial research).
# Download requires accepting the license on HuggingFace Hub.
#
# attn_implementation="eager" disables PyTorch's scaled dot-product attention
# (SDPA) backend.  OpenVLA's modeling_prismatic.py does not declare
# _supports_sdpa, so transformers raises AttributeError when it tries to
# enable SDPA automatically.  "eager" bypasses that check entirely.
from transformers import AutoModelForVision2Seq, AutoProcessor

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model in INT8 (may take 10-20 min on first run)...")
torch.cuda.reset_peak_memory_stats()

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    attn_implementation="eager",
    load_in_8bit=True,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

peak_mem_load_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)
print(f"Model loaded. Peak GPU memory at load: {peak_mem_load_mib:.0f} MiB")

In [ ]:
# ── Cell 10: Model diagnostics — validate auto class, predict_action, unnorm_key
# Run this immediately after loading; it raises informative errors rather than
# letting KeyError or AttributeError surface inside the evaluation loop later.

print("── Model diagnostics ─────────────────────────────────────────────────────")

# 1. Confirm the resolved class looks like an OpenVLA model.
model_class = type(model).__name__
print(f"Loaded class    : {model_class}")
_vla_keywords = ("openvla", "action", "vla")
if any(kw in model_class.lower() for kw in _vla_keywords):
    print("  ✓  Class name contains an OpenVLA keyword — looks correct.")
else:
    print(f"  ⚠  WARNING: '{model_class}' doesn't look like an OpenVLA class.")
    print("     Expected something like 'OpenVLAForActionPrediction'.")
    print("     If predict_action is missing below, try:")
    print("       from transformers import AutoModel")
    print("       model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, ...)")

# 2. Confirm predict_action is available — fail loudly before the eval loop.
if not (hasattr(model, "predict_action") and callable(model.predict_action)):
    raise RuntimeError(
        f"model.predict_action is not available (model class: {model_class}).\n"
        "The model loaded but does not expose the OpenVLA action-prediction API.\n"
        "Possible causes:\n"
        "  • trust_remote_code=True was not applied\n"
        "  • AutoModelForVision2Seq resolved to the wrong class\n"
        "Fix: try loading with AutoModel instead of AutoModelForVision2Seq."
    )
print("predict_action  : available ✓")

# 3. Validate UNNORM_KEY against model.config.norm_stats.
if not (hasattr(model.config, "norm_stats") and model.config.norm_stats):
    print(f"\n  ⚠  model.config.norm_stats not found or empty.")
    print(f"     predict_action will likely fail on unnorm_key='{UNNORM_KEY}'.")
    print("     Inspect model.config directly: print(vars(model.config))")
else:
    available_keys = list(model.config.norm_stats.keys())
    print(f"\nAvailable unnorm keys ({len(available_keys)} total):")
    for k in available_keys:
        tag = "  ← UNNORM_KEY" if k == UNNORM_KEY else ""
        print(f"  '{k}'{tag}")

    if UNNORM_KEY not in available_keys:
        bridge_candidates = [k for k in available_keys if "bridge" in k.lower()]
        hint = (
            f"Likely bridge key(s): {bridge_candidates}"
            if bridge_candidates
            else f"No 'bridge' key found. Full list: {available_keys}"
        )
        raise KeyError(
            f"\n\nUNNORM_KEY='{UNNORM_KEY}' is not in model.config.norm_stats.\n"
            f"{hint}\n\n"
            f"Fix: update UNNORM_KEY in the config cell to one of the listed values "
            f"and re-run from that cell."
        )
    print(f"\n  ✓  UNNORM_KEY='{UNNORM_KEY}' confirmed present in norm_stats.")

print("──────────────────────────────────────────────────────────────────────────")

In [ ]:
# ── Cell 10: Parameter count and hardware profile ─────────────────────────────
import bitsandbytes as bnb_module

n_params = sum(p.numel() for p in model.parameters())
n_params_int8 = sum(
    p.numel() for p in model.parameters()
    if hasattr(p, 'quant_state') or p.dtype == torch.int8
)

print(f"Total parameters : {n_params:,} ({n_params / 1e9:.3f}B)")
print(f"INT8 parameters  : {n_params_int8:,} ({n_params_int8 / n_params * 100:.1f}%)")
print(f"Peak mem (load)  : {peak_mem_load_mib:.0f} MiB")

# Build hardware profile — saved with results for §6 Resource Declaration
props = torch.cuda.get_device_properties(0)
hardware_profile = {
    "gpu_name": props.name,
    "gpu_vram_gb": round(props.total_memory / 1024 ** 3, 2),
    "gpu_multi_proc": props.multi_processor_count,
    "cuda_version": torch.version.cuda,
    "driver_version": driver_ver,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "bitsandbytes_version": bnb_module.__version__,
    "platform": platform.platform(),
    "n_params_total": n_params,
    "n_params_b": round(n_params / 1e9, 3),
    "peak_mem_load_mib": round(peak_mem_load_mib, 1),
}

# Persist hardware profile for Phase 7 and the report
hw_path = Path("configs/hardware_profile.yaml")
with open(hw_path, "w") as f:
    yaml.dump(hardware_profile, f, default_flow_style=False)
print(f"Hardware profile written to {hw_path}")

In [ ]:
# ── Cell 11: Dataset helpers ──────────────────────────────────────────────────

def _find_image_key(obs_dict: dict) -> str:
    """Return the first matching image key from an observation dict."""
    for key in ["image_0", "agentview_rgb", "hand_image", "rgb_static", "rgb", "image"]:
        if key in obs_dict:
            return key
    raise KeyError(
        f"No image key found in observation. Available keys: {list(obs_dict.keys())}"
    )


def _find_action(step_dict: dict) -> np.ndarray:
    """Extract 7-dim action array from a step dict (handles RLDS variants)."""
    if "action" in step_dict:
        act = np.asarray(step_dict["action"], dtype=np.float32)
        return act[:7] if act.ndim == 1 and len(act) >= 7 else act.ravel()[:7]
    # Some OXE datasets split action into sub-keys
    keys = ["world_vector", "rotation_delta", "gripper_closedness_action"]
    parts = [np.asarray(step_dict[k], dtype=np.float32).ravel() for k in keys if k in step_dict]
    if parts:
        return np.concatenate(parts)[:7]
    raise KeyError(f"No action key found in step. Available: {list(step_dict.keys())}")


def episode_to_sample(ep_np: dict) -> dict:
    """
    Extract the first timestep from an RLDS episode (already numpy).

    RLDS episodes loaded via as_numpy_iterator() arrive as:
      ep_np['steps'] = dict of arrays, first axis = time.
    """
    steps = ep_np["steps"]
    obs = steps["observation"]

    img_key = _find_image_key(obs)
    img_arr = obs[img_key][0]  # first timestep, shape (H, W, 3), uint8

    action_gt = _find_action({"action": steps["action"][0]})

    lang_raw = steps.get("language_instruction", [b"pick up the object"])[0]
    lang = lang_raw.decode("utf-8") if isinstance(lang_raw, bytes) else str(lang_raw)
    if not lang.strip():
        lang = "pick up the object"

    return {
        "image": Image.fromarray(img_arr),
        "language": lang,
        "action_gt": action_gt,
    }


print("Dataset helpers defined.")

In [ ]:
# ── Cell 12: Load bridge_dataset from OXE (GCS) ───────────────────────────────
# The OXE GCS bucket is publicly readable with Google credentials (authenticated above).
# Download streams on-demand — we never pull the full dataset, only the episodes we need.

print(f"Connecting to {GCS_DATASET_PATH} ...")
builder = tfds.builder_from_directory(GCS_DATASET_PATH)
print(f"Dataset: {builder.info.name} v{builder.info.version}")
print(f"Features:\n{builder.info.features}")

ds_train = builder.as_dataset(split="train")
print("\nDataset ready. Episodes will be streamed on-demand per seed.")

In [ ]:
# ── Cell 13: Warm-up inference pass ──────────────────────────────────────────
# Prime the GPU kernels so the first timed run is not penalised by JIT compilation.

print("Running warm-up inference (3 forward passes)...")

# Pull one real sample for warm-up
for ep_np in ds_train.take(1).as_numpy_iterator():
    warmup_sample = episode_to_sample(ep_np)
    break

warmup_inputs = processor(
    warmup_sample["language"], warmup_sample["image"]
).to("cuda:0")

with torch.no_grad():
    for _ in range(3):
        _ = model.predict_action(**warmup_inputs, unnorm_key=UNNORM_KEY, do_sample=False)
torch.cuda.synchronize()

torch.cuda.reset_peak_memory_stats()  # reset after warm-up so baseline is clean
print("Warm-up complete. Peak memory stats reset.")

In [ ]:
# ── Cell 14: Timed inference helper ──────────────────────────────────────────

def run_inference_timed(
    image: Image.Image,
    language: str,
    handle,
) -> tuple:
    """
    Run one forward pass and return:
      (action_pred, elapsed_ms, peak_mem_mib, avg_power_w)

    Power is sampled immediately before and after the forward pass;
    the average is used as the representative power for that sample.
    """
    torch.cuda.reset_peak_memory_stats()

    inputs = processor(language, image).to("cuda:0")

    pwr_before_w = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.no_grad():
        action_pred = model.predict_action(
            **inputs, unnorm_key=UNNORM_KEY, do_sample=False
        )

    torch.cuda.synchronize()
    t1 = time.perf_counter()

    pwr_after_w = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0

    elapsed_ms   = (t1 - t0) * 1000.0
    peak_mem_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)
    avg_power_w  = (pwr_before_w + pwr_after_w) / 2.0

    return action_pred, elapsed_ms, peak_mem_mib, avg_power_w


print("Inference helper defined.")

In [ ]:
# ── Cell 15: Main evaluation loop (3 seeds × 200 episodes) ───────────────────
# Each seed shuffles the dataset differently → independent episode subsets.
# PyTorch RNG is also seeded for full reproducibility.

results_per_seed = {}

for seed in SEEDS:
    print(f"\n{'─' * 60}")
    print(f"Seed {seed}  |  evaluating {N_EVAL_EPISODES} episodes")
    print(f"{'─' * 60}")

    torch.manual_seed(seed)
    np.random.seed(seed)

    ds_eval = (
        ds_train
        .shuffle(buffer_size=5000, seed=seed, reshuffle_each_iteration=False)
        .take(N_EVAL_EPISODES)
    )

    l1_errors        = []
    times_ms         = []
    peak_mems_mib    = []
    power_readings_w = []
    skipped          = 0

    wall_t0 = time.perf_counter()

    for ep_idx, ep_np in enumerate(
        tqdm(ds_eval.as_numpy_iterator(), total=N_EVAL_EPISODES, desc=f"seed={seed}")
    ):
        try:
            sample = episode_to_sample(ep_np)
        except (KeyError, IndexError, ValueError) as exc:
            print(f"  episode {ep_idx}: skipped ({exc})")
            skipped += 1
            continue

        action_pred, t_ms, mem_mib, pwr_w = run_inference_timed(
            sample["image"], sample["language"], gpu_handle
        )

        action_gt = sample["action_gt"]
        # Align length: pred is (7,); gt may be longer in some RLDS variants
        min_len = min(len(action_pred), len(action_gt))
        l1 = float(np.abs(action_pred[:min_len] - action_gt[:min_len]).mean())

        l1_errors.append(l1)
        times_ms.append(t_ms)
        peak_mems_mib.append(mem_mib)
        power_readings_w.append(pwr_w)

    wall_elapsed_s = time.perf_counter() - wall_t0
    avg_pwr_w      = float(np.mean(power_readings_w)) if power_readings_w else 0.0
    total_kwh      = (avg_pwr_w * wall_elapsed_s) / 3.6e6
    co2e_g         = total_kwh * 386.0  # US average: 0.386 kg CO2e/kWh

    seed_result = {
        "seed": seed,
        "n_samples": len(l1_errors),
        "n_skipped": skipped,
        "l1_error_mean": float(np.mean(l1_errors)),
        "l1_error_std":  float(np.std(l1_errors)),
        "inference_time_ms_mean": float(np.mean(times_ms)),
        "inference_time_ms_std":  float(np.std(times_ms)),
        "peak_mem_mib_mean": float(np.mean(peak_mems_mib)),
        "peak_mem_mib_std":  float(np.std(peak_mems_mib)),
        "avg_power_w":  avg_pwr_w,
        "wall_time_s":  round(wall_elapsed_s, 2),
        "total_kwh":    round(total_kwh, 6),
        "co2e_g":       round(co2e_g, 4),
    }
    results_per_seed[str(seed)] = seed_result

    print(f"  L1 error       : {seed_result['l1_error_mean']:.4f} ± {seed_result['l1_error_std']:.4f}")
    print(f"  Inference time : {seed_result['inference_time_ms_mean']:.1f} ± {seed_result['inference_time_ms_std']:.1f} ms")
    print(f"  Peak mem       : {seed_result['peak_mem_mib_mean']:.0f} MiB")
    print(f"  Avg power      : {avg_pwr_w:.1f} W")
    print(f"  Energy         : {total_kwh * 1000:.3f} Wh  |  CO2e: {co2e_g:.2f} g")
    print(f"  Wall-clock     : {wall_elapsed_s / 60:.1f} min  ({skipped} skipped)")

print("\nAll seeds complete.")

In [ ]:
# ── Cell 16: Aggregate across seeds ──────────────────────────────────────────
# §5.5 requires mean ± std across at least 3 runs.

def _agg(key):
    vals = [results_per_seed[str(s)][key] for s in SEEDS]
    return {"mean": float(np.mean(vals)), "std": float(np.std(vals)), "n_runs": len(SEEDS), "values": vals}

aggregate = {
    "l1_error":            _agg("l1_error_mean"),
    "inference_time_ms":   _agg("inference_time_ms_mean"),
    "peak_mem_mib":        _agg("peak_mem_mib_mean"),
    "avg_power_w":         _agg("avg_power_w"),
    "total_kwh":           _agg("total_kwh"),
    "co2e_g":              _agg("co2e_g"),
    "wall_time_s":         _agg("wall_time_s"),
}

print("Aggregate across 3 seeds")
print(f"  L1 error       : {aggregate['l1_error']['mean']:.4f} ± {aggregate['l1_error']['std']:.4f}")
print(f"  Inference time : {aggregate['inference_time_ms']['mean']:.1f} ± {aggregate['inference_time_ms']['std']:.1f} ms")
print(f"  Peak mem       : {aggregate['peak_mem_mib']['mean']:.0f} MiB")
print(f"  Total energy   : {aggregate['total_kwh']['mean'] * 1000:.3f} Wh/run")

In [ ]:
# ── Cell 17: Save results/baseline_metrics.json ───────────────────────────────

output = {
    "phase": 1,
    "model": MODEL_ID,
    "model_variant": "INT8_baseline",
    "unnorm_key": UNNORM_KEY,
    "seeds": SEEDS,
    "n_eval_episodes_per_run": N_EVAL_EPISODES,
    "hardware": hardware_profile,
    "per_seed": results_per_seed,
    "aggregate": aggregate,
}

out_path = RESULTS_DIR / "baseline_metrics.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved → {out_path}")

# ── Verification checklist (§ plan Phase 1 Verification) ─────────────────────
print("\n── Verification ─────────────────────────────────────────────")

n_params_check = output["hardware"]["n_params_total"]
ok_params = 6e9 < n_params_check < 9e9
print(f"[{'OK' if ok_params else 'FAIL'}] Param count ~7.5B : {n_params_check/1e9:.3f}B")

peak_mem = aggregate["peak_mem_mib"]["mean"]
vram_limit_mib = hardware_profile["gpu_vram_gb"] * 1024
ok_mem = peak_mem < vram_limit_mib
print(f"[{'OK' if ok_mem else 'FAIL'}] Peak mem < VRAM   : {peak_mem:.0f} MiB < {vram_limit_mib:.0f} MiB")

required_keys = ["l1_error", "inference_time_ms", "peak_mem_mib", "avg_power_w", "total_kwh"]
ok_keys = all(k in aggregate for k in required_keys)
print(f"[{'OK' if ok_keys else 'FAIL'}] All 5 metric types present")

ok_runs = all(aggregate[k]["n_runs"] >= 3 for k in required_keys)
print(f"[{'OK' if ok_runs else 'FAIL'}] n_runs >= 3 for all metrics")

print("────────────────────────────────────────────────────────────")

In [ ]:
# ── Cell 18: Download results from Colab ─────────────────────────────────────
# Downloads both result files so you can commit them to the repo.
from google.colab import files

files.download(str(RESULTS_DIR / "baseline_metrics.json"))
files.download("configs/hardware_profile.yaml")
print("Files downloaded. Commit them to the repo:")
print("  git add results/baseline_metrics.json configs/hardware_profile.yaml")
print("  git commit -m '[phase1] add INT8 baseline metrics and hardware profile'")

## Results Summary

After running all cells, `results/baseline_metrics.json` contains the following
structure (values filled in by the evaluation run):

```
aggregate:
  l1_error          mean ± std  (n_runs=3)   ← §5.5 Compression vs. Accuracy baseline
  inference_time_ms mean ± std  (n_runs=3)   ← §5.5 Efficiency Gain baseline
  peak_mem_mib      mean ± std  (n_runs=3)   ← §5.5 Latency on Reference Profile
  avg_power_w       mean ± std  (n_runs=3)   ← §4.2 Energy baseline
  total_kwh         mean ± std  (n_runs=3)   ← §4.2 Energy baseline
```

These numbers are the **denominator** for Phase 3's efficiency gain and compression
vs. accuracy calculations.  Phase 2 (TN compression) and Phase 3 (evaluation)
read this file as their reference point.

### Challenge sections satisfied
- **§5.4** Accepted baseline (INT8 bitsandbytes) established
- **§5.5** Compression vs. Accuracy baseline: `l1_error` mean ± std
- **§5.5** Efficiency Gain baseline: `inference_time_ms` mean ± std
- **§5.5** Latency on Reference Profile: `inference_time_ms` mean on stated GPU
- **§5.2** Quantitative baseline for the 3-run mean ± std requirement
- **§6**   Hardware and energy data captured in `hardware_profile.yaml`